# Practice 3 — обучение BC на Google Colab (GPU)

Ноутбук для **CPU-потока**: датасеты вы собрали локально в Docker (`dataset/train_1k`, `dataset/train_10k`, `dataset/eval`), а тяжёлое обучение визуомоторной BC-модели делаете здесь, на бесплатной GPU.

**Перед запуском:** меню `Runtime → Change runtime type → Hardware accelerator → GPU`.

Порядок: проверить GPU → склонировать проект → поставить зависимости → подложить датасеты → обучить `bc_1k` и `bc_10k` → посмотреть TensorBoard → прогнать rollout → скачать чекпоинты.

In [ ]:
!nvidia-smi

## 1. Клонируем проект

In [ ]:
!git clone https://github.com/Yandex-Practicum/ap-physiclaai-3.git
%cd ap-physiclaai-3

## 2. Зависимости

PyTorch на Colab уже установлен. Для обучения нужны `timm`, `einops`, `tensorboard`; для rollout-оценки — `mujoco`.

In [ ]:
!pip -q install timm einops tensorboard mujoco

## 3. Датасеты

Нужны папки `dataset/train_1k`, `dataset/train_10k`, `dataset/eval` (с файлами `.npz`).

**Вариант А — через Google Drive (рекомендуется).** Локально заархивируйте папку `dataset/` и загрузите архив в Drive (например, в `MyDrive/practice3/`). Затем выполните ячейку ниже и распакуйте.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Подставьте свой путь. Пример, если вы загрузили dataset.zip:
# !unzip -q /content/drive/MyDrive/practice3/dataset.zip -d .
# Или копирование готовой папки dataset/:
!mkdir -p dataset
!cp -r "/content/drive/MyDrive/practice3/dataset/." dataset/
!echo 'train_1k:' $(ls dataset/train_1k 2>/dev/null | wc -l) ' eval:' $(ls dataset/eval 2>/dev/null | wc -l)

**Вариант Б — готовые артефакты курса** (если преподаватель выдал рабочий `ARTIFACTS_URL`):

In [ ]:
# %env ARTIFACTS_URL=https://storage.googleapis.com/<ваш-бакет>/artifacts
# !python3 scripts/download_artifacts.py --datasets train_1k train_10k eval

## 4. Обучение bc_1k и bc_10k

Чекпоинты сохраняются в `logs/<exp>/checkpoints/{best,last}.pt`, логи TensorBoard — в `logs/<exp>/`.

In [ ]:
!python3 train_bc.py --train_dir dataset/train_1k --eval_dir dataset/eval --exp_name bc_1k --epochs 100 --batch_size 64 --lr 1e-4

In [ ]:
!python3 train_bc.py --train_dir dataset/train_10k --eval_dir dataset/eval --exp_name bc_10k --epochs 100 --batch_size 64 --lr 1e-4

## 5. TensorBoard

Сравните `train/loss` и `eval/loss` для обоих экспериментов.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 6. Rollout-оценка (Success Rate)

MuJoCo на Colab работает в headless-режиме через EGL — задаём `MUJOCO_GL=egl`.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_1k/checkpoints/best.pt  --model bc --episodes 50 --seed 999
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_10k/checkpoints/best.pt --model bc --episodes 50 --seed 999
!MUJOCO_GL=egl python3 inference.py --checkpoint checkpoints/rl_expert.pt        --model rl --episodes 50 --seed 999

## 7. Скачать обученные чекпоинты

Архивируем `logs/bc_1k` и `logs/bc_10k` (веса + логи) и скачиваем на свой компьютер. Дальше их можно положить в локальный проект в те же папки и запускать `inference.py`.

In [ ]:
import shutil
from google.colab import files
for n in ['bc_1k', 'bc_10k']:
    shutil.make_archive(n, 'zip', 'logs', n)
    files.download(f'{n}.zip')